In [ ]:
!pip install ipywidgets tqdm==4.67.1 transformers==4.57.1 bitsandbytes==0.48.1 accelerate==1.11.0 peft==0.17.1 scikit-learn==1.7.2

In [19]:
import torch
import numpy as np
import random
import torchvision
import torchvision.transforms as transforms
from transformers import AutoModelForImageClassification, BitsAndBytesConfig
import torch.nn as nn
from peft import LoraConfig, get_peft_model
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score
import os
from peft import PeftModel

# 시드 고정
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# cuda 사용 가능한지 판단
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='cuda')

In [20]:
# 데이터셋 다운로드
# train 50,000장, test 10,000장
trainset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=False)
testset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=False)

print(trainset.data.shape, testset.data.shape)

100%|██████████| 170M/170M [00:11<00:00, 14.3MB/s] 


(50000, 32, 32, 3) (10000, 32, 32, 3)


In [21]:
# CIFAR_10 데이터셋의 평균과 표준편차 계산

data_tensor = torch.from_numpy(trainset.data)

# 평균 계산 (배치, 높이, 너비 차원에 대해)
mean = torch.mean(data_tensor.float() / 255.0, dim=(0, 1, 2))

# 표준편차 계산 (배치, 높이, 너비 차원에 대해)
std = torch.std(data_tensor.float() / 255.0, dim=(0, 1, 2))

print("CIFAR-10 학습 데이터의 픽셀별 평균:", mean)
print("CIFAR-10 학습 데이터의 픽셀별 표준편차:", std)

CIFAR-10 학습 데이터의 픽셀별 평균: tensor([0.4914, 0.4822, 0.4465])
CIFAR-10 학습 데이터의 픽셀별 표준편차: tensor([0.2470, 0.2435, 0.2616])


In [22]:
# facebook/deit-tiny-patch16-224 모델 input size 인 224*224 로 resize, 정규화(normalize)

# augmentation : 데이터 증강
train_transform_aug = transforms.Compose([
    # 32*32 이미지 주변에 4픽셀의 padding 추가 후, 무작위로 32*32 를 잘라냄
    transforms.RandomCrop(32, padding=4),
    # 50% 확률로 이미지를 좌우로 뒤집기
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

In [23]:
# train / test 를 사용해 만들어진 transform 적용해서 trainset_aug 생성하고, testset 갱신
trainset_aug = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform_aug)
testset  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)

In [ ]:
# 256 batch 로 묶은 데이터로더 생성
trainloader_aug = torch.utils.data.DataLoader(trainset_aug, batch_size=512, shuffle=True, num_workers=4, pin_memory=True)
testloader  = torch.utils.data.DataLoader(testset, batch_size=512, shuffle=False, num_workers=4, pin_memory=True)

print("훈련 배치 개수:", len(trainloader_aug), "테스트 배치 개수:", len(testloader))

훈련 배치 개수: 196 테스트 배치 개수: 40


In [25]:
# QLoRA 를 위해 4비트 양자화된 모델을 불러오기
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,           # 4비트 양자화 활성화
    bnb_4bit_quant_type="nf4",   # 4비트 NormalFloat 사용 (QLoRA 권장)
    bnb_4bit_compute_dtype=torch.float16 # 계산 시 사용할 데이터 타입 설정
)

model_name = "facebook/deit-tiny-patch16-224"

model = AutoModelForImageClassification.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print(model)

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear4bit(in_features=192, out_features=192, bias=True)
              (key): Linear4bit(in_features=192, out_features=192, bias=True)
              (value): Linear4bit(in_features=192, out_features=192, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear4bit(in_features=192, out_features=192, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear4bit(in_features=192, out_features=768, bias=True)
  

- 맨 마지막 레이어는 `classifier` 이다. 원래 출력이 1,000 이었으나, CIFAR-10 에 맞게 10으로 줄일 것

In [26]:
model.classifier = nn.Linear(in_features=model.classifier.in_features, out_features=10)

In [27]:
# LoRA 파라미터 설정
lora_config = LoraConfig(
    # DeiT의 내부 구조를 파악한 후, query, key 에만 LoRA 계층 추가 (이름이 query, key 인지 반드시 확인)
    target_modules=["query", "key",], 
    
    # LoRA Rank (모델의 학습 가능한 파라미터 수)
    r=16, 
    
    # LoRA 스케일링 계수 (r에 대한 학습 속도 조정)
    lora_alpha=16,
    
    # 드롭아웃 확률 (과적합 방지)
    lora_dropout=0.1,
    
    # 'CAUSAL_LM', 'SEQ_CLS' 등 작업에 따라 설정합니다. 분류 작업이므로 'SEQ_CLS'로 설정합니다.
    task_type="SEQ_CLS",
)

model = get_peft_model(model, lora_config)

model = model.to(device)

model.print_trainable_parameters()

trainable params: 149,386 || all params: 5,675,732 || trainable%: 2.6320


- for loop 를 사용해 동결할 가중치를 따로 지정할 필요는 없다.
- `get_peft_model` 사용 시, LoRA 계층만 학습 (현재는 약 2.63 퍼센트)
- QLoRA 는 이미 적용된 것이다. 4비트 양자화된 모델에 LoRA 를 넣으면 QLoRA 이기 때문

In [29]:
# train

# Early Stopping 파라미터 정의
best_val_loss = float('inf')
patience = 5
patience_counter = 0
max_grad_norm = 1.0 # 기울기 클리핑 임계값 정의

# loss function - 다중 클래스 분류에는 Cross Entropy 사용
criterion = nn.CrossEntropyLoss()

# LoRA 사용시엔 AdamW 를 사용하는게 좋습니다.
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

# epoch 설정 (권장: 20 ~ 30)
num_epochs = 30

# 학습률 스케줄러
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=num_epochs)

MODEL_STATE_DICT_DIR_PATH = "./models_state_dict"

os.makedirs(MODEL_STATE_DICT_DIR_PATH, exist_ok=True)

for epoch in tqdm(range(num_epochs)):
    # 훈련모드 지정
    model.train()
    # epoch 가 진행되면서 손실이 줄어드는 과정 확인용
    running_loss = 0.0
    
    for inputs, labels in trainloader_aug:
        inputs, labels = inputs.to(device), labels.to(device)
        
        # 이전 배치의 그래디언트가 다음 배치에 영향을 주지 않도록 초기화
        optimizer.zero_grad()

        with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
            outputs = model(pixel_values=inputs).logits
            loss = criterion(outputs, labels)
        
        # 역전파
        loss.backward()

        # 기울기 클리핑 적용
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        
        # 가중치 업데이트
        optimizer.step()
        
        # 현재 배치의 손실 누적
        running_loss += loss.item()
    
    avg_train_loss = running_loss / len(trainloader_aug)

    # Early Stopping 을 위한 검증 단계 시작
    model.eval()
    val_loss = 0.0
    all_preds = []  # 모든 예측값을 저장할 리스트
    all_labels = []  # 모든 실제 라벨을 저장할 리스트
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
                outputs = model(pixel_values=inputs).logits
            
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            # 예측값을 얻고, 실제 라벨을 저장
            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    avg_val_loss = val_loss / len(testloader)

    # 정확도 계산
    val_accuracy = accuracy_score(all_labels, all_preds)

    print(
        f"[Fine-tune Epoch {epoch+1}/{num_epochs}] "
        f"평균 훈련 손실: {avg_train_loss:.4f}, "
        f"평균 검증 손실: {avg_val_loss:.4f}, "
        f"검증 정확도: {val_accuracy*100:.2f}%, "
        f"현재 학습률: {optimizer.param_groups[0]['lr']:.6f}"
    )

    # Early Stopping
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # 최적의 모델 저장
        model.save_pretrained(MODEL_STATE_DICT_DIR_PATH)
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early Stopping: 검증 손실이 {patience} epoch 동안 개선되지 않아 훈련을 중단합니다.")
            break

    scheduler.step()

  0%|          | 0/30 [00:00<?, ?it/s]

[Fine-tune Epoch 1/30] 평균 훈련 손실: 1.6863, 평균 검증 손실: 0.8610, 검증 정확도: 74.56%, 현재 학습률: 0.000100


  3%|▎         | 1/30 [02:39<1:17:01, 159.37s/it]

[Fine-tune Epoch 2/30] 평균 훈련 손실: 0.6161, 평균 검증 손실: 0.4560, 검증 정확도: 86.10%, 현재 학습률: 0.000100


  7%|▋         | 2/30 [05:15<1:13:23, 157.26s/it]

...

[Fine-tune Epoch 28/30] 평균 훈련 손실: 0.2048, 평균 검증 손실: 0.2177, 검증 정확도: 92.33%, 현재 학습률: 0.000002


 97%|█████████▋| 29/30 [1:15:22<02:35, 155.81s/it]

[Fine-tune Epoch 29/30] 평균 훈련 손실: 0.2067, 평균 검증 손실: 0.2177, 검증 정확도: 92.34%, 현재 학습률: 0.000001


 97%|█████████▋| 29/30 [1:17:58<02:41, 161.32s/it]

[Fine-tune Epoch 30/30] 평균 훈련 손실: 0.2018, 평균 검증 손실: 0.2177, 검증 정확도: 92.32%, 현재 학습률: 0.000000
Early Stopping: 검증 손실이 5 epoch 동안 개선되지 않아 훈련을 중단합니다.


- 저장된 모델의 가중치를 불러와 적용해보자.

In [30]:
# 저장된 모델 가중치 불러와 적용하기

# 1. 동일한 모델 구조 생성
model_name = "facebook/deit-tiny-patch16-224"

# 양자화 제외
loaded_model = AutoModelForImageClassification.from_pretrained(
    model_name,
    device_map="auto"
)

# 2. 마지막 Linear 레이어 수정 (학습 때와 동일하게)
# 이 부분이 중요합니다. 저장할 때와 로드할 때의 레이어 구조가 같아야 합니다.
loaded_model.classifier = nn.Linear(in_features=loaded_model.classifier.in_features, out_features=10)

# 3. 저장된 가중치 로드

loaded_model = PeftModel.from_pretrained(
    loaded_model,
    MODEL_STATE_DICT_DIR_PATH,
    device_map="auto"
)

loaded_model.to(device)

# 4. eval
loaded_model.eval()
correct = 0
total = 0

# 평가할때는 그래디언트 계산 비활성화
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = loaded_model(pixel_values=inputs).logits
        _, predicted = torch.max(outputs, dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

fine_tune_acc = 100 * correct / total
print(f"fine-tuning 후 테스트 정확도: {fine_tune_acc:.2f}%")

fine-tuning 후 테스트 정확도: 92.31%
